In [37]:
# IMPORTS
from pynq import Overlay, allocate, PL, MMIO
from pynq.lib.video import VideoMode
import ipywidgets as widgets
from IPython.display import display
import cv2
import numpy as np
from enum import Enum
import time
import os

projectDir = "/home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator"
PL.reset()
overlay = Overlay(projectDir + "/VivadoHW.bit")
overlay.download()

In [38]:
# SETUP AND DEFINES
IN_WIDTH, IN_HEIGHT = 1280, 720
OUT_WIDTH, OUT_HEIGHT = 1280, 720
IMAGES_DIR = projectDir + "/images"
VALID_IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')
available_images = [os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR) if f.lower().endswith(VALID_IMAGE_EXTENSIONS)] if os.path.exists(IMAGES_DIR) else []
tpg = overlay.v_tpg_0
vdma = overlay.dma_block.axi_vdma_0
dma = overlay.dma_block.axi_dma_0
vdma.writechannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
buttons = overlay.axi_gpios.axi_gpio_0
imageResolution = overlay.axi_gpios.axi_gpio_1
threshold_gpio = overlay.axi_gpios.axi_gpio_2
image_widget = widgets.Image(format='jpeg', width=IN_WIDTH, height=IN_HEIGHT)
stream_switch = overlay.orb_pipeline.axis_switch_0
current_ps_frame = None

class VideoSource(Enum):
    HDMI = (0, 0)
    TPG_COLOR_BAR_BOX = (1, 1)
    TPG_GRID_BOX = (2, 1)
    PS_TO_PL = (3, 2)

ps_to_pl_selected_image = 0
selected_source = None

In [39]:
threshold_gpio.channel1.write(50, 0xFFFFFFFF)
dummy_desc_buf = allocate(shape=(65536,), dtype=np.uint32) 

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
ref_data = []

def extract_hw_reference(img_path):
    print(f"Extraction de la référence matérielle pour {os.path.basename(img_path)}...")
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, (IN_WIDTH, IN_HEIGHT))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    
    if vdma.readchannel.running: vdma.readchannel.stop()
    if vdma.writechannel.running: vdma.writechannel.stop()
    
    tpg.write(0x0000, 0x0)
    stream_switch.write(0x040, 2) 
    stream_switch.write(0x000, 0x2)
    
    vdma.readchannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
    vdma.writechannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
    vdma.readchannel.start()
    vdma.writechannel.start()
    
    buf = vdma.writechannel.newframe()
    np.copyto(buf, img_rgb)
    vdma.writechannel.writeframe(buf)
    
    if dma.recvchannel.idle:
        dma.recvchannel.transfer(dummy_desc_buf)
        
    _ = vdma.readchannel.readframe() 
    
    while not dma.recvchannel.idle:
        pass
        
    transferred_bytes = dma.read(0x58)
    nbrOfDescriptors = transferred_bytes // 64
    
    vdma.readchannel.stop()
    vdma.writechannel.stop()
    
    actual_corners = max(0, nbrOfDescriptors - 1)
    if actual_corners > 0:
        raw_data = dummy_desc_buf[:actual_corners * 16].reshape(actual_corners, 16)
        hw_desc = raw_data[:, :8].copy().view(np.uint8).reshape(actual_corners, 32)
        coords = raw_data[:, 8]
        x_coords = coords & 0xFFFF
        y_coords = (coords >> 16) & 0xFFFF
        kp = [cv2.KeyPoint(x=float(x), y=float(y), size=31.0) for x, y in zip(x_coords, y_coords)]
        
        print(f" -> Succès : {actual_corners} descripteurs FPGA capturés.")
        return {'name': os.path.basename(img_path), 'kp': kp, 'des': hw_desc, 'img': cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)}
    
    print(" -> Échec : Aucun descripteur trouvé.")
    return None

ref_search_path = os.path.join(IMAGES_DIR, "*_ref*")
for img_path in glob.glob(ref_search_path):
    ref_dict = extract_hw_reference(img_path)
    if ref_dict:
        ref_data.append(ref_dict)

current_bounding_box = None
current_object_name = ""

def process_and_rearm_dma():
    global current_desc_count, frame_count, current_bounding_box, current_object_name
    
    status = dma.read(0x34)
    
    # Auto-guérison : Réinitialise le DMA des descripteurs s'il a été coupé par un changement de source
    if (status & 0x1) or (status & 0x10):
        dma.write(0x30, 0x4)
        while dma.read(0x30) & 0x4: pass
        dma.recvchannel.start()
        dma.recvchannel.transfer(dummy_desc_buf)
        return
        
    if (status & 0x2):
        transferred_bytes = dma.read(0x58)
        nbrOfDescriptors = transferred_bytes // 64 
        
        if nbrOfDescriptors > 0:
            actual_corners = max(0, nbrOfDescriptors - 1)
            
            if actual_corners > 0:
                raw_data = dummy_desc_buf[:actual_corners * 16].reshape(actual_corners, 16)
                hw_desc_uint8 = raw_data[:, :8].copy().view(np.uint8).reshape(actual_corners, 32)
                
                coords = raw_data[:, 8]
                x_coords = coords & 0xFFFF
                y_coords = (coords >> 16) & 0xFFFF
                
                current_desc_count = actual_corners
                print(f"[Frame {frame_count}] Detected Corners: {actual_corners}")
                
                kp_live = [cv2.KeyPoint(x=float(x), y=float(y), size=31.0) for x, y in zip(x_coords, y_coords)]
                current_bounding_box = None
                
                if len(ref_data) > 0 and len(kp_live) == actual_corners:
                    best_ref = ref_data[0]
                    matches = bf.match(best_ref['des'], hw_desc_uint8)
                    good_matches = sorted(matches, key=lambda x: x.distance)[:50]
                    
                    if len(good_matches) >= 4:
                        src_pts = np.float32([best_ref['kp'][m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
                        dst_pts = np.float32([kp_live[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
                        H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
                        
                        if H is not None:
                            h, w = best_ref['img'].shape
                            corners_ref = np.float32([[0, 0], [0, h - 1], [w - 1, h - 1], [w - 1, 0]]).reshape(-1, 1, 2)
                            current_bounding_box = cv2.perspectiveTransform(corners_ref, H)
                            current_object_name = best_ref['name']
        
        dma.recvchannel.transfer(dummy_desc_buf)

def setupVideoSource(new_src: VideoSource):
    global selected_source, ps_to_pl_selected_image, current_ps_frame  
    if selected_source == new_src and new_src != VideoSource.PS_TO_PL:
        return

    tpg.write(0x0000, 0x0)
    stream_switch.write(0x040, 0x80000000)
    stream_switch.write(0x000, 0x2)

    # REMARQUE CRITIQUE : Le canal de lecture est arrêté, mais vdma.writechannel N'EST PLUS ARRÊTÉ ICI.
    if vdma.readchannel.running: vdma.readchannel.stop()

    selected_source = new_src
    imageResolution.channel1.write(IN_HEIGHT, 0xFFFFFFFF)
    imageResolution.channel2.write(IN_WIDTH, 0xFFFFFFFF)
    tpg.write(0x0010, IN_HEIGHT)
    tpg.write(0x0018, IN_WIDTH)

    if selected_source == VideoSource.TPG_COLOR_BAR_BOX:
        tpg.write(0x0020, 9); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)
    elif selected_source == VideoSource.TPG_GRID_BOX:
        tpg.write(0x0020, 11); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)

    if not dma.recvchannel.running: dma.recvchannel.start()
    if dma.recvchannel.idle: dma.recvchannel.transfer(dummy_desc_buf)

    if not vdma.writechannel.running:
        vdma.writechannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
        vdma.writechannel.start()

    vdma.readchannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
    vdma.readchannel.start()
    vdma.readchannel.irqframecount = 1

    # Le Switch AXI bascule ici, ouvrant la voie physique aux données
    stream_switch.write(0x040, selected_source.value[1])
    stream_switch.write(0x000, 0x2)

    if selected_source in [VideoSource.TPG_COLOR_BAR_BOX, VideoSource.TPG_GRID_BOX]:
        tpg.write(0x0000, 0x81)
        
    elif selected_source == VideoSource.PS_TO_PL:
        if available_images:
            current_image_path = available_images[ps_to_pl_selected_image]
            print(f"Loading image [{ps_to_pl_selected_image + 1}/{len(available_images)}] for PS_TO_PL: {current_image_path}")
            img = cv2.imread(current_image_path)
            img = cv2.resize(img, (IN_WIDTH, IN_HEIGHT))
            current_ps_frame = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ps_to_pl_selected_image = (ps_to_pl_selected_image + 1) % len(available_images)
            
            # Étant donné que le canal n'a jamais été arrêté et que le switch est ouvert,
            # newframe() est 100% sûr de s'exécuter sans interblocage.
            buf = vdma.writechannel.newframe()
            np.copyto(buf, current_ps_frame)
            vdma.writechannel.writeframe(buf)
        else:
            print("No images available to load.")
            current_ps_frame = None

    print("done")


if not dma.recvchannel.running: dma.recvchannel.start()
dma.recvchannel.transfer(dummy_desc_buf) 

setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
display(image_widget)

print("Streaming via VDMA interrupts...")
old_button_value = buttons.read(0)
frame_count = 0
current_desc_count = 0

try:
    while True:
        process_and_rearm_dma()

        while (vdma.readchannel._mmio.read(0x34) & 0x1000) == 0:
            process_and_rearm_dma()
            
            r_status = vdma.readchannel._mmio.read(0x34)
            if r_status & 0x1: 
                vdma.readchannel.stop()
                vdma.readchannel.start()
                vdma.readchannel.irqframecount = 1
                
            if r_status & 0x8980:
                vdma.readchannel._mmio.write(0x34, 0x8980)
            time.sleep(0.001)

        frame = vdma.readchannel.readframe()
        process_and_rearm_dma()

        vis = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        cv2.putText(vis, f"Descriptors: {current_desc_count}", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
        
        if current_bounding_box is not None:
            vis = cv2.polylines(vis, [np.int32(current_bounding_box)], True, (0, 255, 255), 4, cv2.LINE_AA)
            cv2.putText(vis, f"Object: {current_object_name}", (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
        
        _, jpeg = cv2.imencode('.jpg', vis)
        image_widget.value = jpeg.tobytes()
        
        button_value = buttons.read(0)
        if button_value != old_button_value and button_value > 0:
            if button_value & 0x1: setupVideoSource(VideoSource.HDMI)
            elif button_value & 0x2: setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
            elif button_value & 0x4: setupVideoSource(VideoSource.TPG_GRID_BOX)
            elif button_value & 0x8: setupVideoSource(VideoSource.PS_TO_PL)
        
        old_button_value = button_value
        frame_count += 1
        
except KeyboardInterrupt:
    print("\nStopping stream...")
finally:
    tpg.write(0x00, 0x0)
    if vdma.readchannel.running: vdma.readchannel.stop()
    if vdma.writechannel.running: vdma.writechannel.stop()
    dummy_desc_buf.close()
    print("Stream stopped cleanly.")

done


Image(value=b'', format='jpeg', height='720', width='1280')

Streaming via VDMA interrupts...
[Frame 0] Detected Corners: 3
[Frame 1] Detected Corners: 3
[Frame 1] Detected Corners: 3
[Frame 2] Detected Corners: 6
[Frame 3] Detected Corners: 8
[Frame 4] Detected Corners: 8
[Frame 4] Detected Corners: 4
[Frame 5] Detected Corners: 4
[Frame 5] Detected Corners: 4
[Frame 6] Detected Corners: 4
[Frame 6] Detected Corners: 4
[Frame 7] Detected Corners: 4
[Frame 7] Detected Corners: 4
[Frame 8] Detected Corners: 4
[Frame 8] Detected Corners: 4
[Frame 9] Detected Corners: 8
[Frame 10] Detected Corners: 8
[Frame 11] Detected Corners: 8
[Frame 12] Detected Corners: 8
[Frame 12] Detected Corners: 4
[Frame 13] Detected Corners: 4
[Frame 13] Detected Corners: 4
[Frame 14] Detected Corners: 4
[Frame 14] Detected Corners: 4
[Frame 15] Detected Corners: 4
[Frame 15] Detected Corners: 4
[Frame 16] Detected Corners: 4
[Frame 16] Detected Corners: 4
[Frame 17] Detected Corners: 4
[Frame 17] Detected Corners: 4
[Frame 18] Detected Corners: 4
[Frame 18] Detected Co

In [ ]:
def debug_pipeline():
    print("=== PIPELINE HARDWARE STATUS ===")
    
    # 1. DMA Status (r_brief descriptors)
    print("\n--- DMA (S2MM - Descriptors) ---")
    dma_status = dma.read(0x34)
    print(f"Status Register: {hex(dma_status)}")
    if dma_status & 0x1: print(" [!] HALTED - DMA is stopped.")
    if dma_status & 0x2: print(" [*] IDLE - Waiting for data.")
    if dma_status & 0x10: print(" [FATAL] INTERNAL ERROR (0x5011) - Buffer overflow or TLAST mismatch!")
    if dma_status & 0x20: print(" [FATAL] SLAVE ERROR")
    if dma_status & 0x40: print(" [FATAL] DECODE ERROR")
    if not (dma_status & 0x71): print(" [+] RUNNING - Processing descriptors normally.")

    # 2. VDMA Read Status (Video to PS)
    print("\n--- VDMA Read Channel (S2MM - Video to PS) ---")
    vdma_r_status = vdma.readchannel._mmio.read(0x34)
    print(f"Status Register: {hex(vdma_r_status)}")
    if vdma_r_status & 0x1: print(" [!] HALTED - Read channel is stopped.")
    if vdma_r_status & 0x10: print(" [FATAL] INTERNAL ERROR - AXI stream crashed.")
    if vdma_r_status & 0x80: print(" [!] SOF EARLY ERROR - Frame started before previous finished.")
    if vdma_r_status & 0x100: print(" [!] EOL EARLY ERROR - Line ended too soon (Width mismatch).")
    if vdma_r_status & 0x4000: print(" [!] SOF LATE ERROR - New frame didn't start when expected.")
    if not (vdma_r_status & 0x4191): print(" [+] RUNNING - Capturing video normally.")

    # 3. VDMA Write Status (PS to Video)
    print("\n--- VDMA Write Channel (MM2S - PS to PL) ---")
    vdma_w_status = vdma.writechannel._mmio.read(0x04)
    print(f"Status Register: {hex(vdma_w_status)}")
    if vdma_w_status & 0x1: print(" [!] HALTED")
    if vdma_w_status & 0x10: print(" [FATAL] INTERNAL ERROR")
    if not (vdma_w_status & 0x11): print(" [+] RUNNING - Feeding video normally.")

    # 4. Stream Switch Routing
    print("\n--- AXI Stream Switch ---")
    routing = stream_switch.read(0x040)
    source_map = {0: "HDMI", 1: "TPG", 2: "PS_TO_PL"}
    print(f"Currently routing Si[{routing}] ({source_map.get(routing, 'UNKNOWN')}) to Broadcaster.")

debug_pipeline()

In [ ]:
overlay.orb_pipeline?
